# Pushshift Neural Baseline: GloVe + BiLSTM

This notebook implements the GloVe + BiLSTM neural baseline for the Pushshift Reddit pipeline under the unified project structure.

It is designed to:
- load the processed Reddit dataset from `Dataset/pushshift/`
- align paths with the shared `Code/artifacts/pushshift/` layout
- train a three-class GloVe + BiLSTM neural baseline
- save reusable experiment artifacts for later comparison with TF-IDF and BERT models
- export evaluation outputs that can support later thesis writing

## 0. Verify required packages

This section checks the main dependencies used in the LSTM experiment.
It helps reduce environment-related issues before training starts.

In [ ]:
import importlib
import subprocess
import sys

def install_if_missing(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
        print(f"✅ {package_name} already installed")
    except ImportError:
        print(f"⬇️ Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

required_packages = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("torch", "torch"),
    ("scikit-learn", "sklearn"),
    ("keras-preprocessing", "keras_preprocessing"),
    ("joblib", "joblib"),
]

for package_name, import_name in required_packages:
    install_if_missing(package_name, import_name)

print("Python executable:", sys.executable)

## 1. Setup and unified paths

The notebook resolves paths relative to the current VSCode notebook location.
Outputs are saved under the same `artifacts/pushshift` structure used by the baseline model.

In [ ]:
# === 1) Setup & Unified Paths ===
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from keras_preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROJECT_ROOT = CODE_DIR.parent

DATASET_DIR = PROJECT_ROOT / "Dataset" / "pushshift"
ARTIFACT_ROOT = CODE_DIR / "artifacts"
DATASET_NAME = "pushshift"
DATASET_ARTIFACT_DIR = ARTIFACT_ROOT / DATASET_NAME

SPLIT_DIR = DATASET_ARTIFACT_DIR / "splits"
PRED_DIR = DATASET_ARTIFACT_DIR / "predictions"
MODEL_DIR = DATASET_ARTIFACT_DIR / "models"
CONFIG_DIR = DATASET_ARTIFACT_DIR / "config"
FIG_DIR = DATASET_ARTIFACT_DIR / "figures"
RESOURCE_DIR = DATASET_ARTIFACT_DIR / "resources"

for folder in [ARTIFACT_ROOT, DATASET_ARTIFACT_DIR, SPLIT_DIR, PRED_DIR, MODEL_DIR, CONFIG_DIR, FIG_DIR, RESOURCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATASET_DIR / "reddit_vader_dataset.csv"
TRAIN_PATH = SPLIT_DIR / "train_60.csv"
VAL_PATH   = SPLIT_DIR / "val_10.csv"
TEST_PATH  = SPLIT_DIR / "test_30.csv"

GLOVE_PATH = CODE_DIR / "glove" / "glove.6B.300d.txt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("CSV_PATH:", CSV_PATH)
print("GLOVE_PATH:", GLOVE_PATH)

RANDOM_STATE = 42
MAX_VOCAB = 50_000
MAX_LEN = 80
EMBED_DIM = 300
HIDDEN_DIM = 128
BATCH_SIZE = 64
EPOCHS = 6
LR = 1e-3
DROPOUT = 0.5
EMBED_DROPOUT = 0.2
EARLY_STOPPING_PATIENCE = 2

LABEL_MAP = {0: "Negative", 1: "Neutral", 2: "Positive"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

sns.set_theme(style="whitegrid")

## 2. Load fixed dataset splits

If the shared train/validation/test CSV files already exist, they are reused.
Otherwise, the notebook creates fixed 60/10/30 stratified splits from the Reddit dataset and saves them.

In [ ]:
from sklearn.model_selection import train_test_split

if TRAIN_PATH.exists() and VAL_PATH.exists() and TEST_PATH.exists():
    train_df = pd.read_csv(TRAIN_PATH)
    val_df   = pd.read_csv(VAL_PATH)
    test_df  = pd.read_csv(TEST_PATH)
    print("Loaded existing fixed splits.")
else:
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"Missing dataset file: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["text", "label"]).copy()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"] != ""]
    df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    train_df, temp_df = train_test_split(
        df,
        test_size=0.4,
        random_state=RANDOM_STATE,
        stratify=df["label"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.75,
        random_state=RANDOM_STATE,
        stratify=temp_df["label"]
    )

    train_df.to_csv(TRAIN_PATH, index=False, encoding="utf-8")
    val_df.to_csv(VAL_PATH, index=False, encoding="utf-8")
    test_df.to_csv(TEST_PATH, index=False, encoding="utf-8")
    print("Created and saved fixed splits.")

print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))

In [ ]:
for split_name, split_df in {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}.items():
    print(f"\n{split_name} label distribution:")
    print(split_df["label"].value_counts().sort_index())

## 3. Text preparation

A light cleaning step is applied before tokenisation.
The goal is to keep the Reddit text usable for sequence modelling without over-processing emotional signals.

In [ ]:
import re

def basic_clean(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

for split_df in [train_df, val_df, test_df]:
    split_df["text_clean"] = split_df["text"].astype(str).apply(basic_clean)

train_df[["text", "text_clean", "label"]].head()

## 4. Tokenisation and padding

The training texts are used to build the tokenizer vocabulary.
All sequences are then padded to a fixed length for mini-batch training.

In [ ]:
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text_clean"].astype(str).tolist())

train_seq = tokenizer.texts_to_sequences(train_df["text_clean"].astype(str).tolist())
val_seq   = tokenizer.texts_to_sequences(val_df["text_clean"].astype(str).tolist())
test_seq  = tokenizer.texts_to_sequences(test_df["text_clean"].astype(str).tolist())

X_train = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val   = pad_sequences(val_seq,   maxlen=MAX_LEN, padding="post", truncating="post")
X_test  = pad_sequences(test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

y_train = train_df["label"].astype(int).to_numpy()
y_val   = val_df["label"].astype(int).to_numpy()
y_test  = test_df["label"].astype(int).to_numpy()

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)

print("Vocabulary size used:", vocab_size)
print("Train padded shape:", X_train.shape)
print("Validation padded shape:", X_val.shape)
print("Test padded shape:", X_test.shape)

## 5. Load pretrained GloVe embeddings

This step reads the GloVe vectors from the shared `Code/glove` folder.
If the file is missing, the notebook stops early so the issue is explicit.

In [ ]:
if not GLOVE_PATH.exists():
    raise FileNotFoundError(
        f"Missing GloVe file at: {GLOVE_PATH}\n"
        "Please place glove.6B.300d.txt in the Code/glove folder."
    )

embeddings_index = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        values = line.rstrip().split(" ")
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

print("Loaded GloVe vectors:", len(embeddings_index))

## 6. Build the embedding matrix

The tokenizer vocabulary is aligned with the GloVe vectors.
Words not found in GloVe receive small random initial values so they can still be trained.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
embedding_matrix = rng.normal(0, 0.05, size=(vocab_size, EMBED_DIM)).astype("float32")

hits = 0
for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
        hits += 1

coverage = hits / max(1, vocab_size - 1)
print(f"GloVe hits: {hits}")
print(f"Coverage: {coverage:.2%}")

## 7. Build datasets and dataloaders

The processed arrays are wrapped in PyTorch datasets and dataloaders.
This keeps the training loop consistent with the Sentiment140 neural baseline.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TextDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready.")

## 8. Define the BiLSTM model

A bidirectional LSTM is used as the neural baseline.
Compared with the TF-IDF baseline, this model can capture word order and contextual dependencies more directly.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix: np.ndarray, hidden_dim: int = 128, dropout: float = 0.5, embed_dropout: float = 0.2, num_classes: int = 3):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = True

        self.embedding_dropout = nn.Dropout(embed_dropout)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.embedding_dropout(x)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        h = self.dropout(h)
        logits = self.fc(h)
        return logits

model = BiLSTMClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    embed_dropout=EMBED_DROPOUT,
    num_classes=3,
).to(device)

class_counts = np.bincount(y_train, minlength=3)
class_weights = len(y_train) / (3 * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)
print("Class weights:", class_weights.detach().cpu().numpy())

## 9. Train the model

The model is trained with validation monitoring and early stopping.
The best validation checkpoint is saved for later evaluation and comparison.

In [ ]:
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)

            logits = model(Xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            total_loss += loss.item()
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(yb.cpu().numpy())

    avg_loss = total_loss / max(1, len(loader))
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return avg_loss, probs, preds, labels

history = []
best_val_loss = float("inf")
best_path = MODEL_DIR / "pushshift_bilstm_3class.pt"
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for Xb, yb in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / max(1, len(train_loader))
    val_loss, val_probs, val_preds, val_labels = evaluate_loader(model, val_loader, criterion)

    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_macro_f1={val_f1:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
        print(f"  Saved best checkpoint -> {best_path}")
    else:
        patience_counter += 1
        print(f"  No improvement. Early stopping counter: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("  Early stopping triggered.")
            break

## 10. Save the learning curve

The training history is exported as both a CSV file and a figure.
This makes it easier to compare optimisation behaviour across the baseline, LSTM, and BERT models.

In [ ]:
history_df = pd.DataFrame(history)
history_csv_path = RESOURCE_DIR / "lstm_training_history.csv"
history_df.to_csv(history_csv_path, index=False, encoding="utf-8")

plt.figure(figsize=(7, 4.5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pushshift BiLSTM Training Curve")
plt.legend()
plt.tight_layout()

curve_path = FIG_DIR / "lstm_training_curve.png"
plt.savefig(curve_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved training history CSV:", history_csv_path.resolve())
print("Saved training curve:", curve_path.resolve())

## 11. Evaluate on the test set

The best validation checkpoint is reloaded and evaluated on the held-out test set.
Standard metrics and a confusion matrix are generated for reporting in the dissertation.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))

test_loss, test_probs, test_preds, test_labels = evaluate_loader(model, test_loader, criterion)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(accuracy_score(test_labels, test_preds), 4))
print("Test macro F1:", round(f1_score(test_labels, test_preds, average="macro"), 4))

print("\nClassification report:")
print(classification_report(test_labels, test_preds, digits=4, target_names=[LABEL_MAP[i] for i in range(3)]))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    xticklabels=[LABEL_MAP[i] for i in range(3)],
    yticklabels=[LABEL_MAP[i] for i in range(3)],
    linewidths=0.5,
    linecolor="white",
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix: Pushshift BiLSTM")
plt.tight_layout()

cm_path = FIG_DIR / "lstm_confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved confusion matrix figure:", cm_path.resolve())

## 12. Happiness-oriented summary

The model outputs can be aggregated into simple sentiment signals.
This provides a direct bridge from comment-level classification to happiness-index construction.

In [ ]:
pred_score = np.select(
    [test_preds == 0, test_preds == 1, test_preds == 2],
    [-1.0, 0.0, 1.0],
    default=0.0
)

prob_score = test_probs[:, 2] - test_probs[:, 0]

happiness_summary = {
    "share_negative": float((test_preds == 0).mean()),
    "share_neutral": float((test_preds == 1).mean()),
    "share_positive": float((test_preds == 2).mean()),
    "mean_predicted_sentiment_score": float(pred_score.mean()),
    "mean_probability_balance_score": float(prob_score.mean()),
}

happiness_summary

## 13. Export experiment artifacts

All reusable outputs are saved to the unified artifact structure.
These files support later comparison with the TF-IDF baseline and the future BERT model.

In [ ]:
model_config = {
    "model_name": "glove_bilstm_3class",
    "dataset": "pushshift",
    "random_state": RANDOM_STATE,
    "max_vocab": MAX_VOCAB,
    "max_len": MAX_LEN,
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "dropout": DROPOUT,
    "embed_dropout": EMBED_DROPOUT,
    "best_val_loss": float(best_val_loss),
    "class_weights": class_weights.detach().cpu().numpy().tolist(),
}

model_config_path = CONFIG_DIR / "lstm_model_config.json"
with open(model_config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)

happiness_path = CONFIG_DIR / "lstm_happiness_summary.json"
with open(happiness_path, "w", encoding="utf-8") as f:
    json.dump(happiness_summary, f, ensure_ascii=False, indent=2)

tokenizer_path = RESOURCE_DIR / "lstm_tokenizer.joblib"
joblib.dump(tokenizer, tokenizer_path)

pred_df = test_df.copy()
pred_df["text_clean"] = test_df["text_clean"].values
pred_df["y_true"] = test_labels.astype(int)
pred_df["pred_label"] = test_preds.astype(int)
pred_df["prob_negative"] = test_probs[:, 0].astype(float)
pred_df["prob_neutral"] = test_probs[:, 1].astype(float)
pred_df["prob_positive"] = test_probs[:, 2].astype(float)
pred_df["pred_label_name"] = pred_df["pred_label"].map(LABEL_MAP)
pred_df["true_label_name"] = pred_df["y_true"].map(LABEL_MAP)

pred_path = PRED_DIR / "pred_test_lstm_3class.csv"
pred_df.to_csv(pred_path, index=False, encoding="utf-8")

print("Saved best model:", best_path.resolve())
print("Saved tokenizer:", tokenizer_path.resolve())
print("Saved model config:", model_config_path.resolve())
print("Saved happiness summary:", happiness_path.resolve())
print("Saved predictions:", pred_path.resolve())

## 14. Optional error analysis

Misclassified examples can be inspected qualitatively to understand where the neural baseline still struggles.
This is especially useful for mixed emotion, sarcasm, or context-heavy Reddit comments.

In [ ]:
error_df = pred_df[pred_df["y_true"] != pred_df["pred_label"]].reset_index(drop=True)
print("Number of misclassified examples:", len(error_df))
error_df[["text", "true_label_name", "pred_label_name", "prob_negative", "prob_neutral", "prob_positive"]].head(20)

## 15. Short conclusion

This notebook provides the neural baseline for the Pushshift Reddit experiments.
Its outputs are now aligned with the baseline artifact structure and can be compared directly with TF-IDF + Logistic Regression and the future BERT model.